In [38]:
import pandas as pd
import json
import math 

or_data = pd.read_csv("../../Data/original_data.csv", index_col = 0)
dataset_dict = {}
seed = 42
dataset_dict['Origin'] = or_data 
syn_datas = [f"avatarsk5_{seed}", f"avatarsk10_{seed}", f"ctgan_{seed}", f"gaussiancopula_{seed}", f"synthpop_{seed}", f"tvae_{seed}"]
for syn in syn_datas:
    dataset_dict[syn] = pd.read_csv(f"../../Data/{syn}.csv", index_col = 0)
    
mapped_dataset = {}
with open("../../Data/mapping_genes.json", "r") as f:
    genes_mapping = json.load(f)

abnormal_counts = []
nan_counts = []
for k, v in genes_mapping.items():
    if isinstance(v[0], float) and math.isnan(v[0]):
        nan_counts.append([k, v])
        abnormal_counts.append([k, v])
    elif len(v) > 1 and isinstance(v[0], str):
        abnormal_counts.append([k, v])
        genes_mapping[k] = [v[0]]

def ensembl_to_hugo(col, mapping):
    ensembl_id = col.split('.')[0]  # Remove version
    hugo = mapping.get(ensembl_id, [col])  # fallback: leave as is if not found
    return str(hugo[0])

for k in nan_counts:
    genes_mapping.pop(k[0], None)

genes_cols = []
for col in or_data.columns:
    if col.startswith("ENSG"):
        genes_cols.append(col)

symbols_dataset_dict = {}
for tool, data in dataset_dict.items():
    # If some ENSG columns are missing in a dataset, selecting will raise KeyError.
    # Use intersection to be robust.
    present_genes_cols = [c for c in genes_cols if c in data.columns]
    genes_exp_df = data[present_genes_cols]
    # Rename column
    genes_exp_df_mapped = genes_exp_df.copy()
    genes_exp_df_mapped.columns = [ensembl_to_hugo(col, genes_mapping) for col in genes_exp_df_mapped.columns]

    drop_cols = [col for col in genes_exp_df_mapped.columns if col.startswith("ENSG")]
    genes_exp_df_mapped = genes_exp_df_mapped.drop(columns=drop_cols)

    genes_exp_df_mapped_t = genes_exp_df_mapped.T.reset_index()
    genes_exp_df_mapped_t.columns.values[0] = 'Gene'

    # Averaging duplicated genes
    genes_exp_df_mapped_t_grouped = genes_exp_df_mapped_t.groupby("Gene", as_index=False).mean()
    symbols_dataset_dict[tool] = genes_exp_df_mapped_t_grouped

In [40]:
from scipy.stats import ranksums
import numpy as np
for data in dataset_dict.keys():
    angio_sig = ['VEGFA', 'KDR', 'ESM1', 'PECAM1', 'ANGPTL4', 'CD34']
    or_exp_mat = symbols_dataset_dict[data]
    angio_exp_mat = or_exp_mat[or_exp_mat['Gene'].isin(angio_sig)].set_index('Gene').T
    angio_exp = angio_exp_mat.mean(axis = 1)
    pbrm1_patients = dataset_dict[data][dataset_dict[data]['PBRM1']=="MUT"].index.tolist()
    pbrm1_wt_patients = dataset_dict[data][dataset_dict[data]['PBRM1']=="WT"].index.tolist()
    
    pbrm1_expression_mut = angio_exp[pbrm1_patients].values.tolist()
    pbrm1_expression_wt = angio_exp[pbrm1_wt_patients].values.tolist()
    
    stat, pvalue = ranksums(pbrm1_expression_mut, pbrm1_expression_wt)
    log2fc = np.log2(np.mean(pbrm1_expression_mut)/np.mean(pbrm1_expression_wt))
    print(data, pvalue, log2fc)

Origin 6.903837717796336e-08 0.029609094045037098
avatarsk5_42 0.0003717502566525678 0.013931378879181982
avatarsk10_42 0.0001049245225171111 0.01623026257789898
ctgan_42 0.7513215394074239 0.0038365888716423513
gaussiancopula_42 0.0005390195422011693 0.024698258053468442
synthpop_42 0.6841674122479072 0.0006496324780519793
tvae_42 nan nan


/binary/miniforge3/envs/synthetic_data/lib/python3.9/site-packages/scipy/stats/_stats_py.py:9154: RuntimeWarning: invalid value encountered in scalar divide
  z = (s - expected) / np.sqrt(n1*n2*(n1+n2+1)/12.0)
/binary/miniforge3/envs/synthetic_data/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/binary/miniforge3/envs/synthetic_data/lib/python3.9/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [ ]:
import pandas as pd
import numpy as np

def create_comprehensive_toy_data():
    algos = ['Avatars K5', 'Avatars K10', 'CTGAN', 'Gaussian Copula', 'Synthpop', 'TVAE']
    cancers = ['ccRCC', 'Melanoma', 'NSCLC']
    seeds = range(5)
    
    # --- 1. Data cho Technical Panels (WJI & Spearman) ---
    tech_rows = []
    for cancer in cancers:
        for algo in algos:
            # Giả lập: Avatars ổn định hơn CTGAN
            base_wji = 0.85 if 'Avatars' in algo else 0.4
            base_spearman = 0.75 if 'Avatars' in algo else 0.3
            for seed in seeds:
                tech_rows.append({
                    'Cancer': cancer, 'Algorithm': algo, 'Seed': seed,
                    'WJI': np.clip(base_wji + np.random.normal(0, 0.05), 0, 1),
                    'Spearman': np.clip(base_spearman + np.random.normal(0, 0.05), 0, 1)
                })
    df_tech = pd.DataFrame(tech_rows)

    # --- 2. Data cho Biological Panels (Angiogenesis & Immunoproteasome) ---
    bio_rows = []
    pathways = ['Angiogenesis', 'Immunoproteasome']
    for cancer in cancers:
        for pway in pathways:
            # Dữ liệu Real (Ground Truth)
            bio_rows.append({'Cancer': cancer, 'Type': 'Real', 'Pathway': pway, 'Score': np.random.uniform(0.6, 0.8)})
            # Dữ liệu Synthetic (Lấy trung bình của các thuật toán tiêu biểu)
            for algo in algos:
                score = np.random.uniform(0.5, 0.9) if 'Avatars' in algo else np.random.uniform(0.2, 0.5)
                bio_rows.append({'Cancer': cancer, 'Type': algo, 'Pathway': pway, 'Score': score})
    df_bio = pd.DataFrame(bio_rows)

    # --- 3. Data cho Volcano Plots (DEG) ---
    # Giả lập 1000 genes cho mỗi cặp Real/Synthetic
    volcano_rows = []
    for cancer in cancers:
        genes = [f"Gene_{i}" for i in range(100)]
        for gene in genes:
            lfc_real = np.random.normal(0, 2)
            p_real = 10**(-np.random.uniform(0, 5))
            volcano_rows.append({'Cancer': cancer, 'Gene': gene, 'Source': 'Real', 'LFC': lfc_real, 'Pval': p_real})
            # Synthetic (K10 đại diện)
            volcano_rows.append({'Cancer': cancer, 'Gene': gene, 'Source': 'Avatars K10', 
                                'LFC': lfc_real + np.random.normal(0, 0.2), 
                                'Pval': p_real * np.random.uniform(0.8, 1.2)})
    df_volcano = pd.DataFrame(volcano_rows)

    return df_tech, df_bio, df_volcano

df_tech, df_bio, df_volcano = create_comprehensive_toy_data()
print("Dữ liệu kỹ thuật (5 dòng đầu):")
print(df_tech.head())

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ==========================================
# 1. TẠO TOY DATASET (NHƯ ĐÃ THIẾT KẾ)
# ==========================================
np.random.seed(42)
algos = ['Avatars K5', 'Avatars K10', 'CTGAN', 'Gaussian Copula', 'Synthpop', 'TVAE']
cancers = ['ccRCC', 'Melanoma', 'NSCLC']
colors = sns.color_palette("Set2", len(cancers))

# Data Kỹ thuật (WJI & Spearman)
tech_data = []
for cancer in cancers:
    for algo in algos:
        base_wji = 0.8 if 'Avatars' in algo else 0.3
        base_spr = 0.7 if 'Avatars' in algo else 0.2
        for _ in range(5):
            tech_data.append({
                'Cancer': cancer, 'Algorithm': algo, 
                'WJI': np.clip(base_wji + np.random.normal(0, 0.1), 0, 1),
                'Spearman': np.clip(base_spr + np.random.normal(0, 0.1), 0, 1)
            })
df_tech = pd.DataFrame(tech_data)

# Data Sinh học (Angiogenesis & Immunoproteasome)
bio_data = []
for cancer in cancers:
    for pathway in ['Angiogenesis', 'Immunoproteasome']:
        # Real vs Synthetic (K10 đại diện)
        val = np.random.uniform(0.5, 0.8)
        bio_data.append({'Cancer': cancer, 'Source': 'Real', 'Pathway': pathway, 'Score': val})
        bio_data.append({'Cancer': cancer, 'Source': 'Synthetic', 'Pathway': pathway, 'Score': val + np.random.normal(0, 0.05)})
df_bio = pd.DataFrame(bio_data)

# ==========================================
# 2. THIẾT LẬP BỐ CỤC (LAYOUT) 7 PANELS
# ==========================================
fig = plt.figure(figsize=(22, 16))
# 4 hàng, 3 cột
gs = gridspec.GridSpec(4, 3, height_ratios=[1, 0.8, 1, 0.8])

# Hàm vẽ Heatmap Bayesian (dùng cho Panel 2 & 4)
def plot_bayesian_grid(fig, gs_row, metric_name, data_col):
    for i, cancer in enumerate(cancers):
        ax = fig.add_subplot(gs[gs_row, i])
        # Tạo ma trận giả lập Bayesian
        mat = np.random.dirichlet([1]*len(algos), size=len(algos))
        sns.heatmap(mat, annot=True, fmt=".2f", cmap='RdYlGn', cbar=False, 
                    xticklabels=algos, yticklabels=algos, ax=ax, annot_kws={"size": 7})
        ax.set_title(f"P(Better) {metric_name} - {cancer}", fontsize=10)

# --- PANEL 1 & 3: BOXPLOTS (WJI & SPEARMAN) ---
for i, (metric, title) in enumerate([('WJI', 'Panel 1: Composite WJI'), ('Spearman', 'Panel 3: Spearman Correlation')]):
    ax = fig.add_subplot(gs[i*2, 0:2]) # Chiếm 2 cột đầu
    sns.boxplot(data=df_tech, x='Algorithm', y=metric, hue='Cancer', palette='Set2', ax=ax)
    ax.set_title(title, fontsize=14, weight='bold')
    ax.legend(loc='lower right')

# --- PANEL 2 & 4: BAYESIAN HEATMAPS ---
plot_bayesian_grid(fig, 1, 'WJI', 'WJI')
plot_bayesian_grid(fig, 3, 'Spearman', 'Spearman')

# --- PANEL 5: ANGIOGENESIS EXPRESSION ---
ax5 = fig.add_subplot(gs[0, 2])
sns.barplot(data=df_bio[df_bio['Pathway'] == 'Angiogenesis'], x='Cancer', y='Score', hue='Source', ax=ax5)
ax5.set_title("Panel 5: Angiogenesis Score", weight='bold')

# --- PANEL 6: VOLCANO PLOTS (MÔ PHỎNG) ---
ax6 = fig.add_subplot(gs[1:3, 2])
lfc = np.random.normal(0, 1, 100)
pv = -np.log10(np.random.uniform(0, 1, 100))
ax6.scatter(lfc, pv, alpha=0.5, c='grey')
ax6.scatter(lfc[pv>2], pv[pv>2], c='red', label='Overlap DEGs')
ax6.set_title("Panel 6: Volcano (Real vs Syn)", weight='bold')
ax6.set_xlabel("log2 Fold Change")
ax6.set_ylabel("-log10 P-value")

# --- PANEL 7: IMMUNOPROTEASOME ---
ax7 = fig.add_subplot(gs[3, 2])
sns.lineplot(data=df_bio[df_bio['Pathway'] == 'Immunoproteasome'], x='Cancer', y='Score', hue='Source', marker='o', ax=ax7)
ax7.set_title("Panel 7: Immunoproteasome Trend", weight='bold')

plt.tight_layout()
plt.show()